# Hourly LLM Inference Latency: a 60-second tour

3,496 benchmark runs across 7 LLM inference endpoints, collected hourly since July 2026.

The point of this dataset is that **latency is a distribution, not a number**. Free-tier
endpoints have violent tails: the median stays calm while the p99 triples. Every row here
carries p50, p95 and p99 so you can see that happen.

One thing to know before you start: **27.7% of runs failed completely** (rate limits, quota
rejections). Those rows have null latency columns rather than zeros, because a failed run has
no latency to report. pandas skips nulls in aggregations, so the default behaviour is correct.
Had they been stored as `0.0`, every mean you compute would be understated by ~28%.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("lmex_latency.csv", parse_dates=["ts"])

print(f"{len(df):,} runs | {df.ticker.nunique()} endpoints | "
      f"{df.date.min()} to {df.date.max()}")
print(f"failed runs (null latency): {df.ttft_p99.isna().sum():,}")
df.head()

## 1. The mean lies, the percentiles do not

`ttft_tail_ratio` is p99 divided by p50: how much worse your unluckiest request is than a
typical one. A value of 1.0 would mean a perfectly flat distribution. Nothing here is close.

In [ ]:
summary = df.groupby("ticker").agg(
    p50_ms=("ttft_p50", "median"),
    p99_ms=("ttft_p99", "median"),
    tail_ratio=("ttft_tail_ratio", "median"),
    availability=("error_rate", lambda s: 1 - s.mean()),
).sort_values("tail_ratio", ascending=False)

summary.style.format({
    "p50_ms": "{:.0f}", "p99_ms": "{:.0f}",
    "tail_ratio": "{:.2f}x", "availability": "{:.1%}",
})

## 2. Does time of day matter?

Shared free-tier capacity should show load patterns. Grouping by UTC hour puts roughly 150
runs behind each point, which is enough for the shape to mean something (a single hourly p99
is only that hour's worst request, so never read one in isolation).

In [ ]:
hourly = df.groupby("hour_utc")[["ttft_p50", "ttft_p99"]].median()

ax = hourly.plot(figsize=(10, 4), marker="o", linewidth=2)
ax.set_title("TTFT by hour of day (median across all endpoints)")
ax.set_xlabel("hour (UTC)")
ax.set_ylabel("milliseconds")
ax.set_xticks(range(0, 24, 2))
ax.legend(["p50 (typical request)", "p99 (worst request)"])
ax.grid(alpha=0.3)
plt.tight_layout()

## 3. Fast is not the same as available

The interesting tradeoff in free-tier serving. An endpoint can post excellent latency on the
requests it does answer while rejecting most of them. Plotting both axes at once separates
"fast" from "dependable".

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(summary.p99_ms, summary.availability, s=120, zorder=3)

for name, row in summary.iterrows():
    ax.annotate(name, (row.p99_ms, row.availability),
                xytext=(8, 4), textcoords="offset points")

ax.set_xlabel("median TTFT p99 (ms) -- lower is better")
ax.set_ylabel("availability -- higher is better")
ax.set_title("Latency vs availability: top-left is the place to be")
ax.grid(alpha=0.3, zorder=0)
plt.tight_layout()

## Where to go next

- Track a single endpoint over `date` to catch regime changes when a provider alters its serving stack
- Compare `tok_per_sec` against `ttft_p99`: prefill cost and decode speed are separate phenomena
- Use `error_rate` on its own as an availability time series (it is never null)

Collected with [llm-latency-bench](https://pypi.org/project/llm-latency-bench/)
(`pip install llm-latency-bench`) if you want to benchmark your own endpoints the same way.
Collection code: [github.com/saksham10arora-dotcom/lmex](https://github.com/saksham10arora-dotcom/lmex)